# Assignment 5 — Many-Objective Optimisation with JUSTICE (Local Run)

**Course:** EPA141A Model-Based Decision Making — Delft University of Technology  
**Model:** JUSTICE  

---

## Learning Outcomes

After completing this assignment you will be able to:
1. Load and inspect an optimisation configuration from a JSON file produced in Assignment 4.
2. Run a many-objective evolutionary algorithm on the JUSTICE model via `run_optimization_local.py`.
3. Load and inspect the Pareto-front CSV files produced by a completed MOEA run.

---

## Background

Assignment 4 defined the optimisation problem: 244 RBF decision variables, four objectives, and epsilon values that control archive resolution. This assignment runs the actual Pareto search.

The optimiser is **GenerationalBorg**, a self-adaptive MOEA that simultaneously applies six variation operators (SBX, PCX, DE, UNDX, SPX, UM) and continuously adjusts their selection probabilities based on which generates the most archive improvement. This makes it well suited to high-dimensional spaces like the 244-parameter RBF search space; however, due to the use of many operators, may be slower.  Explore what other algorithms are available in the [ema_workbench optimization docs](https://emaworkbench.readthedocs.io/en/latest/ema_documentation/em_framework/optimization.html) and [Platypus](https://platypus.readthedocs.io/en/latest/api/platypus.algorithms.html), and consider whether a different algorithm is more appropriate for your problem. Any algorithm can be passed to the `algorithm=` argument in `run_optimization_local.py`.


Each function evaluation runs the JUSTICE model once with one candidate policy (one set of 244 RBF parameters), returning four objective values. The algorithm accumulates non-dominated solutions in an **epsilon-archive**: a candidate is only archived if it improves on the current archive by at least ε in at least one objective. Smaller ε gives finer Pareto resolution but requires more function evaluations.

Because MOEAs are stochastic, a single run may miss parts of the Pareto front. Running five independent seeds and pooling their archives with epsilon-dominance produces a **combined reference set** that is more complete than any individual seed. This reference set is the starting point for the Pareto and robustness analyses in Assignments 6 and 7.

---

## Starting Point — Your Assignment 4 configuration

In Assignment 4 you defined the multi-objective optimisation problem and saved your choices to `config/config_student.json`. This notebook loads that config and runs the optimisation using `run_optimization_local.py`, which first needs your input. 


---

## Step 1 — Inspect the Configuration

The optimisation is governed by `config/config_student.json`.  
Read and print each key with an explanation.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, sys, json
import numpy as np
import pandas as pd

# Locate JUSTICE-main (one level up from model_answers_ema/)
try:
    _NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _NOTEBOOK_DIR = os.path.abspath('.')
_JUSTICE_ROOT = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "../JUSTICE-main"))

if _JUSTICE_ROOT not in sys.path:
    sys.path.insert(0, _JUSTICE_ROOT)

os.chdir(_JUSTICE_ROOT)

# ── Inspect config ────────────────────────────────────────────────────────────
CONFIG_PATH = os.path.join(_NOTEBOOK_DIR, "../config/config_student.json")

with open(CONFIG_PATH) as fh:
    cfg = json.load(fh)

explanations = {
    "start_year":                       "Simulation start year",
    "end_year":                         "Simulation end year",
    "data_timestep":                    "Years between raw input data points",
    "timestep":                         "Model integration timestep (years)",
    "emission_control_start_year":      "First year ECR can exceed zero",
    "n_rbfs":                           "Number of RBFs (effective: n_inputs + 2)",
    "n_inputs":                         "RBF input signals ",
    "epsilons":                         "Archive granularity",
    "temperature_year_of_interest":     "Year for threshold fraction evaluation",
    "reference_ssp_rcp_scenario_index": "Reference scenario index",
}

print(f"Configuration: {CONFIG_PATH}\n")
for k, v in cfg.items():
    print(f"  {k:<40s}  {str(v):<15}  # {explanations.get(k, '')}")

In [ ]:
from justice.util.data_loader import DataLoader

# Decision variable count
n_inputs   = cfg["n_inputs"]
n_rbfs_act = n_inputs + 2
n_regions  = len(DataLoader().REGION_LIST)

n_centers = n_rbfs_act * n_inputs
n_radii   = n_rbfs_act * n_inputs
n_weights = n_rbfs_act * n_regions
n_total   = n_centers + n_radii + n_weights

print("RBF decision variable summary:")
print(f"  n_rbfs_actual = {n_rbfs_act}  (n_inputs + 2 = {n_inputs} + 2)")
print(f"  n_regions     = {n_regions}")
print(f"  ─────────────────────────────────")
print(f"  Centers       = {n_rbfs_act} × {n_inputs} = {n_centers:4d}  range [-1,  1]")
print(f"  Radii         = {n_rbfs_act} × {n_inputs} = {n_radii:4d}  range [ 0,  1]")
print(f"  Weights       = {n_rbfs_act} × {n_regions} = {n_weights:4d}  range [ 0,  1]")
print(f"  ─────────────────────────────────")
print(f"  TOTAL                    = {n_total}")

print(f"\nEpsilon values per objective:")
obj_names = ["welfare", "fraction_above_threshold", "welfare_loss_damage", "welfare_loss_abatement"]
for name, eps in zip(obj_names, cfg["epsilons"]):
    print(f"  {name:<35s}  ε = {eps}")

---

## Step 2 — Run the Optimisation

The optimisation is implemented in `run_optimization_local.py`. Go through it
and make sure that your configuration is properly integrated!  Update it with your own modeling choices.  This is very important. You can run the current python code as is, to get a feel, but make the adjustments necessary 
for your particular problem formulation.
Open a **terminal** in the `assignments_ema/` directory and run:

```bash
# Quick smoke-test (~3-5 min, 1 seed, 500 NFE) highly suggested before a fully fledged run
python run_optimization_local.py --nfe 500 --seeds 9845531

# Full single-seed run (~1-3 h)
python run_optimization_local.py --nfe 50000 --seeds 9845531

# Full 5-seed run (background, ~1 working day)
nohup python run_optimization_local.py --nfe 50000 > opt_log.txt 2>&1 &
```

You can also launch it from the notebook cell below after you complete it (it blocks the kernel until finished, it may take a while):

In [ ]:
# ── CONFIGURE HERE ───────────────────────────────────────────────────────────
NFE         = # YOUR NUMBER OF FUNCTION EVALUATIONS
SEEDS       =  #YOUR CODE run at least 5 random seeds, eg. 1,2,3..
N_ENSEMBLES = # YOUR CODE            #  FAIR members (local speed)
N_PROCESSES = None         # None → auto-detect (cpu_count - 1)
OUTPUT_DIR  = os.path.join(_NOTEBOOK_DIR, "results")
# ─────────────────────────────────────────────────────────────────────────────

seeds_str  = " ".join(str(s) for s in SEEDS)
n_proc_arg = f"--n_processes {N_PROCESSES}" if N_PROCESSES else ""

# Derive script path from _JUSTICE_ROOT (reliable regardless of kernel CWD)
script_path = os.path.normpath(
    os.path.join(_JUSTICE_ROOT, "..", "assignments_ema", "run_optimization_local.py")
)
cmd = (
    f"python {script_path} "
    f"--nfe {NFE} "
    f"--seeds {seeds_str} "
    f"--n_ensembles {N_ENSEMBLES} "
    f"--output_dir {OUTPUT_DIR} "
    f"--config " + os.path.join(_NOTEBOOK_DIR, "../config/config_student.json") + " "
    + n_proc_arg
)

print("Command to run:")
print(f"  {cmd}")
print("\nRunning ... (this cell blocks until complete)")

ret = os.system(cmd)
print(f"\nExit code: {ret}  ({'OK' if ret == 0 else 'ERROR'})")

---

## Step 3 — Load and Inspect Results

Each completed seed produces a directory e.g.  `<welfare_function>_<nfe>_<seed>/` inside `results/` containing:
- `pareto_front_<seed>.csv` — the final Pareto-optimal solutions (levers + objectives).
- `<welfare_function>_<nfe>_<seed>.tar.gz` — the ArchiveLogger convergence history (used in Assignment 6).
- `convergence_<seed>.csv` — EpsilonProgress and operator probabilities per NFE checkpoint (used in Assignment 6).

Load all available Pareto front CSVs and print a statistical summary.  
>Tip: you can use print(all_results[OBJECTIVE_COLS].describe().round(n))
Print the nfes, and number of solutions in the pareto front as well. 

In [ ]:
import glob

# Path to results (assignments_ema/results/)
RESULTS_ROOT = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "results"))

#YOUR CODE HERE


In [ ]:
# Verify results are non-trivial:
# A non-trivial Pareto front should have:
#   - More than 1 solution
#   - Some variation in all four objective columns
#   - fraction_above_threshold < 1.0 for at least some solutions
#     (meaning the MOEA found policies better than BAU)

#YOUR CODE HERE

---

## Reflection Questions

**1. Multiple seeds.** Why do we run the MOEA with 5 different random seeds rather than running a single long seed for 5× the NFE? What property of the algorithm makes this necessary?

**2. NFE size.**  What diagnostic would tell you whether the NFE you tried was enough?

**3. Operator adaptation.** GenerationalBorg continuously adapts the selection probabilities of its six variation operators (SBX, PCX, DE, UNDX, SPX, UM) during the run. What does this mean and why is it useful for the 244-dimensional RBF search space?  What is a key tradeoff of this MOEA's adaptive learning feature?


---

## Appendix — Script Reference

```
run_optimization_local.py — full argument list

  --nfe           int     NFE per seed                 (default: 50000)
  --seeds         int+    Random seeds                 (default: all 5)
  --output_dir    str     Results root directory        (default: answers_ema/results/)
  --n_processes   int     Worker processes              (default: auto)
  --n_ensembles   int     FAIR ensemble members         (default: 10)
  --config        str     JSON config path              (default: config/config_student.json)
  --population    int     MOEA population size          (default: 100)
```

**Output directory structure:**

```
assignments_ema/
└── results/
    └── <welfare_function>_<nfe>_<seed>/
        ├── <welfare_function>_<nfe>_<seed>.tar.gz   ← convergence archive (Assignment 6)
        ├── convergence_<seed>.csv            ← EpsilonProgress + operator probs (Assignment 6)
        └── pareto_front_<seed>.csv           ← final Pareto front
```

For example, running with `--nfe 50000 --seeds 9845531` produces:

```
assignments_ema/
└── results/
    └── <welfare_function>_50000_9845531/
        ├── <welfare_function>_50000_9845531.tar.gz
        ├── convergence_9845531.csv
        └── pareto_front_9845531.csv
```